<a href="https://colab.research.google.com/github/Suyash-28-commits/Reinforcement-Learning/blob/main/Dual_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch transformers

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM , AutoTokenizer
import random
import numpy as np

In [ ]:
from transformers import BitsAndBytesConfig

In [ ]:
from collections import deque

In [ ]:
import torch.optim as optim

In [ ]:
USE_8BIT = True
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
device = "cuda"

In [ ]:
tokenizer =  AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

if not USE_8BIT:
  bnb_config = BitsAndBytesConfig(load_in_8bit = True)
  load_kwargs = {"quantization_config":bnb_config,"device_map":"auto"}
else:
  load_kwargs = {}

generator = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **load_kwargs
)

if not USE_8BIT:
  generator.to(device)
generator.config.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2",device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
EMBED_DIM = 384

In [ ]:
context_segment = [
    "Reinforcement learning allows robots to learn motor skills such as walking, "
    "grasping, and navigation by trial and error. The robot improves through reward-based feedback.",

    "Supervised learning relies on labeled datasets where input-output pairs are "
    "provided during training. Used in image classification, spam detection, and medical diagnosis.",

    "Convolutional neural networks are used in computer vision for object detection, "
    "facial recognition, and image segmentation. They learn spatial hierarchies via convolutions.",

    "Modern robots consist of sensors, actuators, and controllers. Sensors such as "
    "cameras and LiDAR provide environmental information to the robot.",

    "The human brain contains billions of neurons connected through synapses. "
    "Neural activity allows humans to process information, reason, and control movement.",

    "Quantum mechanics describes particle behavior at microscopic scales. "
    "Wave-particle duality and quantum superposition are central concepts.",

    "Natural language processing enables machines to understand and generate human language. "
    "Applications include translation, chatbots, and text summarization.",

    "Deep reinforcement learning combines neural networks with RL to solve complex "
    "decision-making problems such as autonomous driving and robotic manipulation.",

    "Medical imaging techniques such as MRI and CT scans allow doctors to visualize "
    "internal body structures for diagnosis and treatment planning.",

    "Telescopes allow astronomers to observe distant galaxies and stars. "
    "Modern observatories use advanced sensors to capture high-resolution space images.",

]

In [ ]:
input_prompt = "Explain how reinforcement learning benefits robotics."

In [ ]:
NUM_SEGMENTS = len(context_segment)
MAX_SELECTED = 3

In [ ]:
class ContextSelectorDQN(nn.Module):
  def __init__(self,embed_dim : int,num_segments : int):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(embed_dim,256),
        nn.LayerNorm(256),
        nn.ReLU(),
        nn.Linear(256,128),
        nn.ReLU(),
        nn.Linear(128,num_segments)
    )


  def forward(self , x : torch.Tensor) -> torch.Tensor:
    return self.net(x)


In [ ]:
dqn = ContextSelectorDQN(EMBED_DIM,NUM_SEGMENTS).to(device)
dqn_target = ContextSelectorDQN(EMBED_DIM,NUM_SEGMENTS).to(device)
dqn_target.load_state_dict(dqn.state_dict())
dqn_target.eval()

ContextSelectorDQN(
  (net): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [ ]:
optimizer = optim.Adam(dqn.parameters(),lr=3e-4)
criterion = nn.MSELoss()

In [ ]:
Transition = tuple

In [ ]:
class ReplayBuffer():
  def __init__(self,capacity : int = 500):
    self.buf = deque(maxlen=capacity)


  def push(self,*args):
    self.buf.append(Transition(args))

  def sample(self, batch_size: int):
    return random.sample(self.buf,batch_size)

  def __len__(self):
    return len(self.buf)

replay_buffer = ReplayBuffer(capacity=500)


In [ ]:
#Embedding cache
_emb_cache : dict[str,torch.Tensor] = {}

In [ ]:
def get_embedding(text:str) -> torch.Tensor:
  if text not in _emb_cache:
    _emb_cache[text] =  embedder.encode(text,convert_to_tensor=True)
  return _emb_cache[text]

In [ ]:
segment_embeddings = [get_embedding(seg) for seg in context_segment]
query_embedding = get_embedding(input_prompt)

In [ ]:
#Context selection (epsilon = greedy, top-k contexts)
def select_contexts(state: torch.Tensor, epsilon : float = 0.4) -> list[int]:
  if random.random() < epsilon:
    return random.sample(range(NUM_SEGMENTS),MAX_SELECTED)


  with torch.no_grad():
    scores = dqn(state.unsqueeze(0).to(device)).squeeze(0)

  top_k = torch.topk(scores,MAX_SELECTED).indices.tolist()
  return top_k



In [ ]:
def compute_reward(
    answer : str,
    selected_indices : list[int],
    alpha : float = 0.4,
    beta : float = 0.25,
    gamma : float = 0.15,
    delta : float = 0.15,
    eps : float = 0.05) -> float:

    ans_emb = get_embedding(answer)

    #similarity between relevance to query
    rel_query = torch.cosine_similarity(
        ans_emb.unsqueeze(0),query_embedding.unsqueeze(0)
    ).item()

    #relevance between selected contexts (mean)
    ctx_sims = [
        torch.cosine_similarity(
            ans_emb.unsqueeze(0),
            segment_embeddings[i].unsqueeze(0)
        ).item() for i in selected_indices
    ]

    rel_context = float(np.mean(ctx_sims))

    #diversity : averager r pairwise distance between selected segments
    #High diversity -> good (no redundant chunks)
    if len(selected_indices) > 1:
      pairwise = []
      for i in range(len(selected_indices)):
        for j in range(i+1,len(selected_indices)):
          sim = torch.cosine_similarity(
              segment_embeddings[selected_indices[i]].unsqueeze(0),
              segment_embeddings[selected_indices[j]].unsqueeze(0)
          ).item()
          pairwise.append(1.0 -sim)
      diversity = float(np.mean(pairwise))
    else:
      diversity = 0.5 #if only segment

    #dilution penalty
    #A segment is on topic if its cosine similarity to the query > threshold
    RELEVANCE_THRESHOLD = 0.3
    off_topic = sum(
        1 for i in selected_indices
        if torch.cosine_similarity(
            query_embedding.unsqueeze(0),
            segment_embeddings[i].unsqueeze(0)
        ).item() < RELEVANCE_THRESHOLD
    )

    dilution_penalty = off_topic / len(selected_indices)

    #Length bonus
    length_bonus = min(len(answer.split()) /80 ,1.0)

    reward = (
        alpha * rel_query
        +beta*rel_context
        +gamma*diversity
        -delta*dilution_penalty
        +eps*length_bonus
    )

    return float(reward)



In [ ]:
#Updation of DQN
BATCH_SIZE = 16
GAMMA_RL = 0.9 #discount factor for future rewards
TARGET_UPDATE = 5 # syncing of target network every N episodes

In [ ]:
def update_dqn(episode : int):
  if len(replay_buffer) < BATCH_SIZE:
    return None

  batch = replay_buffer.sample(BATCH_SIZE)
  states , actions_list , rewards , next_states = zip(*batch)

  states_t = torch.stack(states).to(device)
  next_states_t = torch.stack(next_states).to(device)
  rewards_t = torch.tensor(rewards,dtype = torch.float32).to(device)

  #Current q values for all segments and gathering selected ones
  q_all = dqn(states_t)

  #Target Q
  with torch.no_grad():
    next_q_all = dqn_target(next_states_t)
    max_next_q = next_q_all.max(dim=1).values


  targets = rewards_t + GAMMA_RL* max_next_q

  losses = []
  for b_idx,selected in enumerate(actions_list):
    for a in selected:
      losses.append(criterion(q_all[b_idx,a],targets[b_idx]))

  loss = torch.stack(losses).mean()

  optimizer.zero_grad()
  loss.backward()
  nn.utils.clip_grad_norm_(dqn.parameters(),max_norm=1.0)
  optimizer.step()


  if episode % TARGET_UPDATE == 0:
    dqn_target.load_state_dict(dqn.state_dict())

  return loss.item()

In [ ]:
#Helper for generator
def generate_answer(prompt: str, max_new_tokens: int = 80) -> str:
  inputs = tokenizer(prompt,return_tensors="pt",truncation=True,max_length=400).to("cuda")
  with torch.no_grad():
    output = generator.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

  full_text = tokenizer.decode(output[0],skip_special_tokens=True)
  answer = full_text[len(prompt):].strip()
  return answer if answer else "No answer generated"


In [ ]:
#training loop
NUM_EPISODES = 20
EPSILON_START = 0.8
EPSILON_END = 0.15
EPSILON_DECAY = (EPSILON_START - EPSILON_END)/ NUM_EPISODES

state = query_embedding.to(device)

In [ ]:
generator.to("cuda") #gpt-2

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
generator.to("cuda") #tinyLLama

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
for ep in range(1,NUM_EPISODES+1):
  epsilon = max(EPSILON_END,EPSILON_START - EPSILON_DECAY *ep)

  #Selected contexts
  selected_indices = select_contexts(state,epsilon=epsilon)
  selected_texts = [context_segment[i] for i in selected_indices]


  context_block = "\n".join(
      f"Context {j+1} : {txt}" for j , txt in enumerate(selected_texts)
  )

  prompt  = f"Question: {input_prompt}\n{context_block}"

  #Generation
  answer = generate_answer(prompt)

  #Reward
  reward = compute_reward(answer,selected_indices)

  #Storing of transition
  replay_buffer.push(state.detach().cpu().clone(),selected_indices,reward, state.detach().cpu().clone())

  #Updation of DQN
  loss = update_dqn(ep)

  loss_str = f"{loss:.4f}" if loss is not None else "N/A"
  print(f"Episode {ep:02d} | epsilon={epsilon:.2f} | Reward={reward:.4f} | Loss={loss_str}")
  print(f"Selected segments: {selected_indices}")
  for j , txt in enumerate(selected_texts):
    print(f"[{j+1}] {txt[:80]}....")

  print(f"Answer : {answer[:200]}")



Episode 01 | epsilon=0.77 | Reward=0.4860 | Loss=0.1931
Selected segments: [3, 5, 0]
[1] Modern robots consist of sensors, actuators, and controllers. Sensors such as ca....
[2] Quantum mechanics describes particle behavior at microscopic scales. Wave-partic....
[3] Reinforcement learning allows robots to learn motor skills such as walking, gras....
Answer : Reinforcement Learning (RL) is a branch of AI that focuses on solving problems using rewards or punishments. RL can be applied in robotics by training a neural network that maps input data into an out
Episode 02 | epsilon=0.74 | Reward=0.3675 | Loss=0.1659
Selected segments: [8, 4, 6]
[1] Medical imaging techniques such as MRI and CT scans allow doctors to visualize i....
[2] The human brain contains billions of neurons connected through synapses. Neural ....
[3] Natural language processing enables machines to understand and generate human la....
Answer : Question: Can you explain the relationship between neural networks and reinfo

In [ ]:
#Inference
print("Final answer:-------")
final_indices = select_contexts(state,epsilon=0.0)
final_texts = [context_segment[i] for i in final_indices]
context_block = "\n".join(f"[Context {j+1}] : {t}" for  j, t in enumerate(final_texts))
prompt = f"Question : {input_prompt} \n {context_block}\nAnswer:"
final_answer = generate_answer(prompt,max_new_tokens=120)
print(f"Selected: {final_indices}")
print(f"Answer:{final_answer}")


Final answer:-------
Selected: [2, 0, 4]
Answer:Reinforcement learning is a powerful tool that enables robots to learn from experience and improve their performance over time. By imitating human behavior, it can perform tasks more effectively than a single algorithm alone. One example is the use of deep reinforcement learning algorithms in robotic arm movements. In this case, the robot learns how to move objects based on rewards provided by the environment. This approach has led to significant improvements in robotic manipulation. Another application of reinforcement learning is in robot navigation. By using a combination of sensors and perception systems, the robot


In [ ]:
def select_context(state,epsilon):

  if random.random() < epsilon:
    return random.randint(0,action_size-1)

  with torch.no_grad():
    q_values = dqn(state)
  action = torch.argmax(q_values).item()

  return action

In [ ]:
# def heirarchical_reasoner(selected_segments,reasoner_model,tokenizer):
#   #Combined segments
#   combined = " ".join(selected_segments)
#   embedding = get_embedding(combined,reasoner_model,tokenizer)

#   #Reward = length-normalized embdedding magnitude
#   reward = embedding.norm().item()/len(selected_segments)
#   return reward , embedding

In [ ]:
#Training loop
num_episodes = 10
for ep in range(num_episodes):
  print(f"------Episode{ep+1}------")

  #Reasoner evaluates each context segment
  state_embeddings = []
  for idx, seg in enumerate(context_segments):

    if idx in reasoner_memory:
      embedding = reasoner_memory[idx]
    else:
      embedding = get_embedding(seg)
      reasoner_memory[idx] = embedding

    state_embeddings.append(embedding)

  state = get_embedding(input_prompt)
  state = state.squeeze().unsqueeze(0).to(device)

  #RL selects context
  action_idx = select_context(state)
  selected_context = context_segments[action_idx]

  #Generates uses context
  prompt = (
      input_prompt
      +"\nContext: "
      +selected_context
      +"\nAnswer: "
  )

  gen_inputs = tokenizer(prompt,return_tensors="pt").to(device)
  gen_output = generator.generate(
      **gen_inputs,
      max_length=120,
      pad_token_id=tokenizer.eos_token_id,
      do_sample=True,
      temperature=0.8,
      top_p=0.9
  )
  gen_text = tokenizer.decode(gen_output[0],skip_special_tokens=True)

  answer = gen_text[len(prompt):].strip()

  if len(answer) == 0:
    answer = "No answer generated"

  #Reasoner computes reward
  reward = compute_reward(answer,selected_context)
  reward = torch.tensor(reward,dtype=torch.float32).to(device)

  #Update DQN
  q_values = dqn(state)
  pred_q = q_values[0][action_idx]


  target =reward.squeeze()
  loss = criterion(pred_q,target)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print("Selected context: ")
  print(selected_context[:150],"...")
  print("\nGenerator output: ")
  print(answer[:200],"...")

  print("\nReward: ",reward.item())

------Episode1------


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x384 and 768x128)

In [ ]:
contexts = [

# Robotics
"Modern robots consist of sensors, actuators, and controllers that allow them to perceive their environment.",

# Medicine
"Medical imaging techniques such as MRI and CT scans help doctors diagnose diseases.",

# Climate
"Climate change refers to long-term shifts in global temperatures and weather patterns.",

# AI
"Deep learning uses neural networks with many layers to learn representations from data.",

# Economics
"Inflation is the rate at which the general level of prices for goods and services rises.",

# Biology
"Cells are the fundamental units of life and contain genetic information in DNA.",

# Astronomy
"Black holes are regions of spacetime where gravity is so strong that nothing can escape.",

# Networking
"The Internet works through packet switching where data is broken into packets and routed."

]

In [ ]:
num_episodes = 20
gamma = 0.9


for ep in range(num_episodes):

    print(f"\n------Episode {ep+1}------")

    # epsilon decay
    epsilon = max(0.1,1-num_episodes/100)


    # state = embedding of question
    state = get_embedding(input_prompt)
    state = state.clone().detach().unsqueeze(0).to(device)

    # select context
    action_idx = select_context(state, epsilon)
    selected_context = context_segments[action_idx]

    # prompt
    prompt = f"""
Answer the question using the context.

Question:
{input_prompt}

Context:
{selected_context}

Answer:
"""

    gen_inputs = tokenizer(prompt, return_tensors="pt").to(device)

    gen_output = generator.generate(
        **gen_inputs,
        max_length=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    gen_text = tokenizer.decode(gen_output[0], skip_special_tokens=True)

    answer = gen_text[len(prompt):].strip()

    if len(answer) == 0:
        answer = "No answer generated"

    # compute reward
    reward = compute_reward(answer, selected_context)
    reward = torch.tensor(reward, dtype=torch.float32).to(device)

    # DQN update
    q_values = dqn(state)

    pred_q = q_values[0][action_idx]

    with torch.no_grad():
        max_q = torch.max(q_values)

    target = reward + gamma * max_q

    loss = criterion(pred_q, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Selected context:")
    print(selected_context[:150], "...")

    print("\nGenerated answer:")
    print(answer[:200], "...")

    print("\nReward:", reward.item())


------Episode 1------
Selected context:
Modern robots consist of sensors, actuators, and controllers that allow them to perceive their environment and perform physical actions. Sensors such  ...

Generated answer:
Robots are not only physically intelligent, they also have the ability to learn and adapt to any environment they choose. The problem with reinforcement learning is that it is not designed to be "lear ...

Reward: 0.6638409495353699

------Episode 2------
Selected context:
Modern robots consist of sensors, actuators, and controllers that allow them to perceive their environment and perform physical actions. Sensors such  ...

Generated answer:
This is the first robot-based solution to create the ability to observe a robot's environment.

This robot-based solution has a range of sensors, including cameras, actuators, and controllers. It can  ...

Reward: 0.6690115332603455

------Episode 3------
Selected context:
Medical imaging techniques such as MRI and CT scans allow docto